In [1]:
import numpy as np
import pandas as pd
import anndata
import scanpy as sc
import scvelo as scv
import multivelo as mv

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
import sys
sys.path.append('../')
from UTV_metrics import *

In [3]:
num_comps = 9


In [4]:
data_outdir = 'processed_data/'
model_outdir = 'modeling_results/'

In [5]:
celltype_name = 'cell_type'

In [6]:
adata_rna = anndata.read_h5ad(data_outdir+'adata_rna.h5ad')

In [ ]:
SCV_result = anndata.read_h5ad(model_outdir+'scvelo_result.h5ad')
MV_result = anndata.read_h5ad(model_outdir+'multivelo_result.h5ad')
AA_result = anndata.read_h5ad(model_outdir+'archvelo_result.h5ad')

In [ ]:
import os
fig_outdir = 'figures/'
os.makedirs(fig_outdir, exist_ok = True)


# Velocity fields

In [ ]:
celltype_name = 'cell_type'

In [ ]:
sns.set(style = 'ticks', font_scale = 1.3)

In [ ]:
scv.pl.velocity_embedding_stream(SCV_result, 
                                 color = celltype_name,
                                 show = False,
                                 title = 'scVelo',
                                fontsize = 25)
plt.savefig(fig_outdir+'scVelo_results.png', dpi = 500, transparent = True)
plt.savefig(fig_outdir+'scVelo_results.svg', dpi = 500, transparent = True)

In [ ]:
mv.velocity_graph(MV_result)
mv.latent_time(MV_result)
mv.velocity_embedding_stream(MV_result, 
                             show=False, 
                             color = celltype_name, 
                             title = 'MultiVelo',
                             fontsize = 25)
plt.savefig(fig_outdir+'MultiVelo_results.png', dpi = 500, transparent = True)
plt.savefig(fig_outdir+'MultiVelo_results.svg', dpi = 500, transparent = True)

In [ ]:
# mv.velocity_graph(AA_result)
# mv.latent_time(AA_result)
mv.velocity_embedding_stream(AA_result, 
                             show=False, 
                             color = celltype_name, 
                             title = 'ArchVelo',
                            fontsize = 25)
plt.savefig(fig_outdir+'AA_results.png', dpi = 500, transparent = True)
plt.savefig(fig_outdir+'AA_results.svg', dpi = 500, transparent = True)

# CBD

In [ ]:
inds = np.array([[i]+list(np.nonzero(x)[1]) for (i,x) in enumerate(adata_rna.obsp['distances'])])
SCV_result.uns['neighbors']['indices'] = inds
MV_result.uns['neighbors']['indices'] = inds
AA_result.uns['neighbors']['indices'] = inds

## finer notation

In [ ]:
edges = [('HSC/MPP', 'LMPP'), 
         ('LMPP', 'GMP'),
         ('GMP', 'Prog N'), 
         ('LMPP', 'Prog DC'),    
         ('HSC/MPP', 'EMP'), 
         ('EMP', 'MEP'),
         ('EMP', 'BEM'),
         ('MEP', 'BEM'),
         ('MEP', 'Ery'),
         ('MEP', 'Prog MK'),
         ('Prog MK', 'Platelet')]
        

In [ ]:
len(edges)

In [ ]:
CBCs = {}

In [ ]:
CBCs['scVelo'] = cross_boundary_correctness(
    SCV_result, 
    celltype_name, 
    'velocity', 
    edges, 
    x_emb="X_umap", return_raw = True)

In [ ]:
CBCs['MultiVelo'] = cross_boundary_correctness(
    MV_result, 
    celltype_name, 
    'velo_s_norm', 
    edges, 
    x_emb="X_umap", return_raw = True)

In [ ]:
CBCs['ArchVelo'] = cross_boundary_correctness(
    AA_result, 
    celltype_name, 
    'velo_s_norm', 
    edges, 
    x_emb="X_umap", return_raw = True)

In [ ]:
sns.set(style = 'ticks', font_scale = 1.2)
plt.figure(figsize = (15,6), dpi = 500)
methods = CBCs.keys()
all_CBCs = pd.concat([pd.concat([pd.Series(CBCs[met][k]) for k in CBCs[met].keys()],
         keys = [str(x[0]) for x in list(zip(CBCs[met].keys()))]) for met in methods],
          keys = methods, axis = 0, join = 'outer').reset_index()
all_CBCs.columns = ['Method', 'Edge', 'Cell','CBD']
sns.barplot(all_CBCs, 
            y = 'CBD',
            x = 'Edge',
            #order = edge_order,
            hue = 'Method', 
             palette = ['#E64F4F', '#506ED8', '#AAAAAA'],#palette = np.array(sns.color_palette('hls',4))[[0,2,3]], #swap_axes = True,
             hue_order = ['ArchVelo', 'MultiVelo', 'scVelo'])#, order = edges)
plt.xticks(ticks = range(len(edges)),labels = [str(x) for x in range(1,1+len(edges))],fontsize = 20)
plt.ylabel('CBDir', fontsize = 20)
plt.xlabel('Edge',fontsize = 20)
plt.legend(frameon = False, fontsize = 20)
plt.ylim(-0.9, 1.2)
plt.savefig(fig_outdir+'CBCs_with_CMP_no_MV_AA.png', dpi = 500, transparent = True)
plt.savefig(fig_outdir+'CBCs_with_CMP_no_MV_AA.svg', dpi = 500, transparent = True)

In [ ]:
from scipy.stats import wilcoxon, ttest_rel
i=0
for edge in edges:
    i+=1
    cur_df = all_CBCs.loc[all_CBCs['Edge'].isin([str(edge)]),:]
    xx = cur_df.query('Method == "ArchVelo"')['CBD']
    yy = cur_df.query('Method == "MultiVelo"')['CBD']
    #if np.abs(np.mean(xx)-np.mean(yy))>0.05:
    print('i=',i)
    print(edge)
    p = wilcoxon(x = xx, 
             y = yy,
             zero_method='wilcox').pvalue
    print(p)
    if p<0.01/len(edges):
        if np.mean(xx)> np.mean(yy):
            print('ArchVelo')
        else:
            print('MultiVelo')

In [ ]:
from scipy.stats import wilcoxon, ttest_rel
i=0
for edge in edges:
    i+=1
    cur_df = all_CBCs.loc[all_CBCs['Edge'].isin([str(edge)]),:]
    xx = cur_df.query('Method == "ArchVelo"')['CBD']
    yy = cur_df.query('Method == "scVelo"')['CBD']
    #if np.abs(np.mean(xx)-np.mean(yy))>0.05:
    print('i=',i)
    print(edge)
    p = wilcoxon(x = xx, 
             y = yy,
             zero_method='wilcox').pvalue
    print(p)
    if p<0.01/len(edges):
        if np.mean(xx)> np.mean(yy):
            print('ArchVelo')
        else:
            print('scVelo')